# 02. Feature Engineering

Gene filtering and pathway aggregation for survival prediction.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, DEFAULT_CANCER_TYPE
from src.data.load_tcga import load_tcga_data
from src.features.gene_filtering import select_variable_genes, apply_gene_filter
from src.features.pathway_aggregation import compute_pathway_scores, add_pathway_features

%matplotlib inline

## 1. Load Data

In [ ]:
merged_df, _ = load_tcga_data(
    clinical_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_clinical.csv",
    expression_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_expression.csv",
    merge=True
)
print(f"Loaded data: {merged_df.shape}")

## 2. Gene Filtering

In [ ]:
# Select top variable genes
selected_genes = select_variable_genes(merged_df, top_n=50)
print(f"Selected {len(selected_genes)} genes")
print(f"Top 10 genes: {selected_genes[:10]}")

In [ ]:
# Apply gene filter
filtered_df = apply_gene_filter(
    merged_df,
    selected_genes,
    keep_cols=['patient_id', 'OS_time', 'OS_status']
)
print(f"Filtered data: {filtered_df.shape}")

## 3. Pathway Aggregation

In [ ]:
# Compute pathway scores
pathway_scores = compute_pathway_scores(merged_df, aggregation='mean')
print(f"Pathway scores: {pathway_scores.shape}")
pathway_scores.head()

In [ ]:
# Visualize pathway scores
pathway_cols = [c for c in pathway_scores.columns if c.startswith('pathway_')]

fig, ax = plt.subplots(figsize=(10, 6))
pathway_scores[pathway_cols].boxplot(ax=ax)
ax.set_ylabel('Pathway Score')
ax.set_title('Pathway Score Distribution')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Add pathway features to expression data
combined_df = add_pathway_features(merged_df)
print(f"Combined features: {combined_df.shape}")

## 4. Feature Correlation

In [ ]:
# Correlation between pathways
corr_matrix = pathway_scores[pathway_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Pathway Correlation Matrix')
plt.tight_layout()
plt.show()

## Summary

- Selected top variable genes
- Computed pathway-level features
- Combined genes and pathways for modeling
- Next: Train survival models